In [1]:
# ライブラリのインポート
import pandas as pd
import numpy as np
import seaborn as sns


In [21]:
def read_data(file_path):
    try:
        df = pd.read_csv(file_path, encoding="utf-8")
    except:
        print("ファイルが見つかりません")
    return df


In [109]:
# 全共通前処理（データ読込→結合・共通の列追加・レコード削除）

file_path_dict = {
    "df_train_match_data" : "data/train_Jleague.csv",
    "df_train_match_add_data" : "data/train_add.csv",
    "df_train_2014_match_add_data" : "data/2014_add.csv",
    "df_stadium_data" : "data/stadium.csv",
    "df_match_condition_data" : "data/condition.csv",
    "df_match_condition_add_data" : "data/condition_add.csv",
    "df_test_data" : "data/test_Jleague.csv"
    }


def read_data_and_join():
    df_data = {}
    for key, value in file_path_dict.items():
        df_data[key] = read_data(value)
    
    # 学習用データと追加データを結合する。
    df_data["train_full_data"] = pd.concat([df_data["df_train_match_data"], df_data["df_train_match_add_data"],])
    df_data["match_full_data"] = pd.concat([df_data["df_match_condition_data"], df_data["df_match_condition_add_data"],])

    df_data["train_full_data"] = df_data["train_full_data"].merge(df_data["match_full_data"], on="id", how="left")
    df_data["df_stadium_data"].rename(columns={"name": "stadium"}, inplace=True)
    df_data["train_full_data"]  = df_data["train_full_data"].merge(df_data["df_stadium_data"], on="stadium", how="left")

    # テストデータも同様に結合する。
    df_data["test_full_data"] = df_data["df_test_data"].merge(df_data["match_full_data"], on="id", how="left")
    df_data["test_full_data"] = df_data["test_full_data"].merge(df_data["df_stadium_data"], on="stadium", how="left")

    # 曜日を抽出する。
    df_data["train_full_data"]["weekday"] = df_data["train_full_data"]["gameday"].str.extract(r'\((.)\)')
    # 土日の場合は1、それ以外は0
    df_data["train_full_data"]["is_weekend"] = df_data["train_full_data"]["weekday"].isin(["土", "日"]).astype(int)
    # 月を抽出する。
    df_data["train_full_data"]["month"] = df_data["train_full_data"]["gameday"].str.split("/").str[0].astype(int)
    # 時間を数値化する
    df_data["train_full_data"]["time"] = df_data["train_full_data"]["time"].str.split(":").str[0].astype(int)
    # ホームチームの平均観客数を追加
    home_avg = df_data["train_full_data"].groupby("home_team")["y"].mean()
    df_data["train_full_data"]["home_team_avg_attendance"] = df_data["train_full_data"]["home_team"].map(home_avg)
    # 湿度を数字データに変換
    df_data["train_full_data"]["humidity"] = df_data["train_full_data"]["humidity"].str.replace("%", "").astype(float)

    # テストデータも同様
    df_data["test_full_data"]["weekday"] = df_data["test_full_data"]["gameday"].str.extract(r'\((.)\)')
    df_data["test_full_data"]["is_weekend"] = df_data["test_full_data"]["weekday"].isin(["土", "日"]).astype(int)
    df_data["test_full_data"]["month"] = df_data["test_full_data"]["gameday"].str.split("/").str[0].astype(int)
    df_data["test_full_data"]["time"] = df_data["test_full_data"]["time"].str.split(":").str[0].astype(int)
    df_data["test_full_data"]["home_team_avg_attendance"] = df_data["test_full_data"]["home_team"].map(home_avg)
    df_data["test_full_data"]["humidity"] = df_data["test_full_data"]["humidity"].str.replace("%", "").astype(float)

    # yが0の試合を除外する
    df_data["train_full_data"] = df_data["train_full_data"].drop(df_data["train_full_data"][df_data["train_full_data"]["y"] == 0].index).reset_index(drop=True)


    return df_data



In [47]:
# LightGBM用に加工する関数。
def convert_for_lgbm(df: pd.DataFrame, datetime_action: str = "decompose") -> pd.DataFrame:
    """
    LightGBMで使えない型を自動変換する。

    Parameters
    ----------
    df : pd.DataFrame
    datetime_action : str
        "decompose" → 年/月/日/曜日に分解（デフォルト）
        "drop"      → datetime列を削除

    Returns
    -------
    pd.DataFrame : 変換済みDataFrame
    """
    df = df.copy()

    for col in df.columns:
        dtype = df[col].dtype

        # 文字列(object) → category
        if dtype == "object":
            df[col] = df[col].astype("category")
            #print(f"[category] {col}: object → category")

        # datetime → 分解 or 削除
        elif pd.api.types.is_datetime64_any_dtype(dtype):
            if datetime_action == "decompose":
                df[f"{col}_year"]    = df[col].dt.year
                df[f"{col}_month"]   = df[col].dt.month
                df[f"{col}_day"]     = df[col].dt.day
                df[f"{col}_weekday"] = df[col].dt.weekday
                df = df.drop(columns=[col])
                print(f"[decompose] {col}: datetime → year/month/day/weekday")
            else:
                df = df.drop(columns=[col])
                print(f"[drop] {col}: datetime列を削除")

        # bool → int（念のため明示変換）
        elif dtype == "bool":
            df[col] = df[col].astype(int)
            print(f"[int] {col}: bool → int")

    return df

In [75]:
# LightGBMの学習

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import lightgbm as lgb

def run_cv( params, gbm_train, feature, target_col, n_splits=5, use_log=False):
    kfold = KFold(n_splits=n_splits, shuffle=True, random_state=0)
    scores = []
    for idx, (train_idx, valid_idx) in enumerate(kfold.split(gbm_train)):
        x_train = gbm_train.loc[train_idx, feature]
        y_train = gbm_train.loc[train_idx, target_col]
        x_valid = gbm_train.loc[valid_idx, feature]
        y_valid = gbm_train.loc[valid_idx, target_col]

        if use_log:
            y_train = np.log1p(y_train)

        train_set = lgb.Dataset(x_train, y_train)
        valid_set = lgb.Dataset(x_valid, np.log1p(y_valid) if use_log else y_valid,
                                reference=train_set)
        model = lgb.train(
            params, train_set,
            num_boost_round=10000,
            valid_sets=[valid_set],
            callbacks=[lgb.early_stopping(stopping_rounds=150)],
        )
        pred = model.predict(x_valid)
        if use_log:
            pred = np.expm1(pred)
        score = np.sqrt(mean_squared_error(y_valid, pred))
        scores.append(score)
        print(f"fold {idx+1} RMSE: {score:.2f}")

    mean_score = np.mean(scores)
    print(f"CV mean RMSE: {mean_score:.2f}")
    return mean_score

In [93]:
# LightGBM用の前処理

# データを読み込んで結合したデータフレームを取得
df = read_data_and_join()
#print(df["train_full_data"].tail(3), df["train_full_data"].shape)
#print(df["test_full_data"].head(3), df["test_full_data"].shape)

# いったん全てのデータを使う想定でやってみる。
#print(df["train_full_data"].dtypes)

df["train_full_data"] = convert_for_lgbm(df["train_full_data"])
df["test_full_data"] = convert_for_lgbm(df["test_full_data"])

#print(df["train_full_data"].dtypes)

# データが準備できたので、学習を行う。
# パラメータの設定
LGB_PARAMS = {
    'boost': 'gbdt',
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.01,
    "max_depth": 8,
    "num_leaves": 128, 
    "lambda_l1": 0.01,
    "lambda_l2": 0.01,
    "bagging_fraction": 0.7,
    "bagging_freq": 5,
    "feature_fraction": 0.8,
    'verbosity': -1,
}

#def run_cv( params, gbm_train, feature, target_col, n_splits=5, use_log=False):

# 全ての列を使ったのは微妙なので、いくつか省いて行う
exclude_cols = ["y", "id", "year", "gameday", "home_score", "away_score"]
feature_cols = [c for c in df["train_full_data"].columns if c not in exclude_cols]


score = run_cv(LGB_PARAMS, df["train_full_data"], df["train_full_data"].columns.drop("y"), "y", use_log=False)




Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[1178]	valid_0's rmse: 2658.53
fold 1 RMSE: 2658.53
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[1105]	valid_0's rmse: 3082.66
fold 2 RMSE: 3082.66
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[1270]	valid_0's rmse: 3211.09
fold 3 RMSE: 3211.09
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[771]	valid_0's rmse: 3047.85
fold 4 RMSE: 3047.85
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[840]	valid_0's rmse: 3147.69
fold 5 RMSE: 3147.69
CV mean RMSE: 3029.56


In [94]:
def run_cv_and_predict(params, gbm_train, x_test, feature, target_col, n_splits=5, use_log=False):
    kfold = KFold(n_splits=n_splits, shuffle=True, random_state=0)
    scores = []
    test_preds = []  # 各foldの予測を溜める

    for idx, (train_idx, valid_idx) in enumerate(kfold.split(gbm_train)):
        x_train = gbm_train.loc[train_idx, feature]
        y_train = gbm_train.loc[train_idx, target_col]
        x_valid = gbm_train.loc[valid_idx, feature]
        y_valid = gbm_train.loc[valid_idx, target_col]

        if use_log:
            y_train = np.log1p(y_train)

        train_set = lgb.Dataset(x_train, y_train)
        valid_set = lgb.Dataset(x_valid, np.log1p(y_valid) if use_log else y_valid,
                                reference=train_set)
        model = lgb.train(
            params, train_set,
            num_boost_round=10000,
            valid_sets=[valid_set],
            callbacks=[lgb.early_stopping(stopping_rounds=150)],
        )

        # バリデーションスコア
        pred_valid = model.predict(x_valid)
        if use_log:
            pred_valid = np.expm1(pred_valid)
        score = np.sqrt(mean_squared_error(y_valid, pred_valid))
        scores.append(score)
        print(f"fold {idx+1} RMSE: {score:.2f}")

        # テストデータを予測して溜める
        pred_test = model.predict(x_test[feature])
        if use_log:
            pred_test = np.expm1(pred_test)
        test_preds.append(pred_test)

    print(f"CV mean RMSE: {np.mean(scores):.2f}")

    # 全foldの予測を平均する
    final_pred = np.mean(test_preds, axis=0)
    return final_pred

In [113]:
# テストデータの結果も一緒に行う
# LightGBM用の前処理

# データを読み込んで結合したデータフレームを取得
df2 = read_data_and_join()
#print(df["train_full_data"].tail(3), df["train_full_data"].shape)
#print(df["test_full_data"].head(3), df["test_full_data"].shape)

# いったん全てのデータを使う想定でやってみる。
#print(df["train_full_data"].dtypes)

df2["train_full_data"] = convert_for_lgbm(df2["train_full_data"])
df2["test_full_data"] = convert_for_lgbm(df2["test_full_data"])

#print(df["train_full_data"].dtypes)

# データが準備できたので、学習を行う。
# パラメータの設定
LGB_PARAMS = {
    'boost': 'gbdt',
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.01,
    "max_depth": 8,
    "num_leaves": 128, 
    "lambda_l1": 0.01,
    "lambda_l2": 0.01,
    "bagging_fraction": 0.7,
    "bagging_freq": 5,
    "feature_fraction": 0.8,
    'verbosity': -1,
}

#def run_cv( params, gbm_train, feature, target_col, n_splits=5, use_log=False):

# 全ての列を使ったのは微妙なので、いくつか省いて行う
exclude_cols = ["y", "year", "referee",  "id", "home_score", "away_score"]

feature_cols = [c for c in df["train_full_data"].columns if c not in exclude_cols]

feature_no_players = [c for c in feature_cols
                      if not (c.startswith("home_0") or 
                              c.startswith("home_1") or
                              c.startswith("away_0") or 
                              c.startswith("away_1"))]

final_pred = run_cv_and_predict(
    LGB_PARAMS,
    df["train_full_data"],
    df["test_full_data"],
    feature_no_players,
    "y",
    use_log=False 
)

# 提出ファイル作成
submission = pd.DataFrame({
    "id": df["test_full_data"]["id"],
    "y": final_pred
})
submission.to_csv("submission_GBM2.csv", index=False, header=False)
print(submission.head())
print(f"予測値の範囲: {final_pred.min():.0f} 〜 {final_pred.max():.0f}")


Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[490]	valid_0's rmse: 2755.55
fold 1 RMSE: 2755.55
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[1025]	valid_0's rmse: 3115.67
fold 2 RMSE: 3115.67
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[922]	valid_0's rmse: 3321.22
fold 3 RMSE: 3321.22
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[610]	valid_0's rmse: 3130.05
fold 4 RMSE: 3130.05
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[825]	valid_0's rmse: 3157.29
fold 5 RMSE: 3157.29
CV mean RMSE: 3095.96
      id             y
0  15822  14691.893283
1  15823  19054.022841
2  15824  33353.377627
3  15825  12601.477058
4  15827  27124.617959
予測値の範囲: 2608 〜 45588


In [97]:
df2["train_full_data"].columns

Index(['id', 'y', 'year', 'stage', 'match', 'gameday', 'time', 'home', 'away',
       'stadium', 'tv', 'home_score', 'away_score', 'weather', 'temperature',
       'humidity', 'referee', 'home_team', 'home_01', 'home_02', 'home_03',
       'home_04', 'home_05', 'home_06', 'home_07', 'home_08', 'home_09',
       'home_10', 'home_11', 'away_team', 'away_01', 'away_02', 'away_03',
       'away_04', 'away_05', 'away_06', 'away_07', 'away_08', 'away_09',
       'away_10', 'away_11', 'address', 'capa', 'weekday', 'is_weekend',
       'month'],
      dtype='object')

In [98]:
feature_no_players = [c for c in feature_cols
                      if not (c.startswith("home_0") or 
                              c.startswith("home_1") or
                              c.startswith("away_0") or 
                              c.startswith("away_1"))]

feature_no_players

['stage',
 'match',
 'time',
 'home',
 'away',
 'stadium',
 'tv',
 'weather',
 'temperature',
 'humidity',
 'referee',
 'home_team',
 'away_team',
 'address',
 'capa',
 'weekday',
 'is_weekend',
 'month']

In [111]:
df2["train_full_data"].to_csv("train_full_data.csv", index=False)